# PyDI Data Integration Workflow: Products

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with product datasets to showcase the data integration pipeline from information extraction over schema and entity matching to data fusion.

## Table of Contents

## Part 1: Information Extraction

In [1]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.informationextraction import LLMExtractor
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

from __future__ import annotations
import os
from collections import Counter
from typing import Any


/work/lucschwa/conda_envs/pydi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1: Load Source Datasets

In [11]:
from PyDI.io import load_json

products_1 = load_json(INPUT_DIR / "data" / "products_1.json")
products_1.attrs["dataset_name"] = "products_1"

products_2 = load_json(INPUT_DIR / "data" / "products_2.json")
products_2.attrs["dataset_name"] = "products_3"

products_3 = load_json(INPUT_DIR / "data" / "products_3.json")
products_3.attrs["dataset_name"] = "products_3"

products_4 = load_json(INPUT_DIR / "data" / "products_4.json")
products_4.attrs["dataset_name"] = "products_4"

products_1.head()

,id,brand,title,description,price,priceCurrency,cluster_id,url
0,12198483,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,"CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...",799.99,GBP,1002037,https://www.novatech.co.uk/products/gigabyte-n...
1,78378158,WD,"WD Blue 6TB 3.5\"" SATA 3 HDD/Hard Drive","6TB WD Blue WD60EZAZ, 3.5\"" HDD, SATA III - 6G...",129.98,GBP,1004942,https://www.scan.co.uk/products/6tb-wd-blue-wd...
2,80641070,None,Corsair Force MP510 M.2 SSD - 960GB,"Solid State Drive, 960 GB, intern, M.2 2280, P...",2124.00,NOK,1007272,https://www.proshop.no/SSD/Corsair-Force-MP510...
3,51886539,Seagate,"Seagate EXOS 4TB 3.5\"" SATA HDD/Hard Drive","4TB Seagate EXOS ST4000NM0115, 3.5\"" Enterpris...",142.99,GBP,1014052,https://www.scan.co.uk/products/4tb-seagate-ex...
4,10328354,None,Asus GeForce GTX 1650 Phoenix OC 4GB Video Card,The ASUS GeForce GTX 1650 Phoenix OC 4GB Video...,219.00,AUD,1014152,https://www.auspcmarket.com.au/video-cards/nvi...


### Step 2: Create Schema for Information Extraction

In [ ]:
from typing import Optional
from pydantic import BaseModel

class ProductSchema(BaseModel):
    brand: Optional[str] = None
    model: Optional[str] = None
    product_type: Optional[str] = None
    core_clock_mhz: Optional[int] = None
    memory_clock_mhz: Optional[int] = None
    adaptive_refresh_rate: Optional[bool] = None
    memory_gb: Optional[int] = None
    interface: Optional[str] = None
    read_speed_mb_s: Optional[int] = None
    write_speed_mb_s: Optional[int] = None


### Step 3: Run LLM-Based Extraction using Schema


In [17]:
from dotenv import load_dotenv
load_dotenv()

# Create single source column containing all information for extraction
for dataset in [products_1, products_2, products_3, products_4]:
    dataset["title_description"] = dataset[["title", "description"]].fillna("").astype(str).agg(". Description: ".join, axis=1)

extractor = LLMExtractor(
    chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0),
    source_column="title_description",
    schema=ProductSchema,
    system_prompt="Extract product information as JSON matching the schema."
)

extracted_products_1 = extractor.extract(products_1)

extracted_products_1.head()

Pydantic validation failed: 1 validation error for ProductSchema
memory_gb
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=1.2, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/int_from_float
Pydantic validation failed: 1 validation error for ProductSchema
memory_gb
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=0.146, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/int_from_float
Pydantic validation failed: 1 validation error for ProductSchema
memory_gb
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=1.2, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/int_from_float
Pydantic validation failed: 1 validation error for ProductSchema
memory_gb
  Input should be a valid integer, got a number with a fractio

,id,brand,title,description,price,priceCurrency,cluster_id,url,title_description,model,product_type,interface,series,memory_gb,core_clock_mhz,memory_clock_mhz,read_speed_mb_s,write_speed_mb_s
0,12198483,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,"CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...",799.99,GBP,1002037,https://www.novatech.co.uk/products/gigabyte-n...,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,NVIDIA GeForce RTX 3080 Gaming OC,None,None,None,10.0,1800.0,19.0,None,None
1,78378158,WD,"WD Blue 6TB 3.5\"" SATA 3 HDD/Hard Drive","6TB WD Blue WD60EZAZ, 3.5\"" HDD, SATA III - 6G...",129.98,GBP,1004942,https://www.scan.co.uk/products/6tb-wd-blue-wd...,"WD Blue 6TB 3.5\"" SATA 3 HDD/Hard Drive. Descr...",WD60EZAZ,None,SATA III,Blue,6.0,NaN,NaN,None,None
2,80641070,Corsair,Corsair Force MP510 M.2 SSD - 960GB,"Solid State Drive, 960 GB, intern, M.2 2280, P...",2124.00,NOK,1007272,https://www.proshop.no/SSD/Corsair-Force-MP510...,Corsair Force MP510 M.2 SSD - 960GB. Descripti...,Force MP510,None,PCI Express 3.0 x4 (NVMe),None,960.0,NaN,NaN,None,None
3,51886539,Seagate,"Seagate EXOS 4TB 3.5\"" SATA HDD/Hard Drive","4TB Seagate EXOS ST4000NM0115, 3.5\"" Enterpris...",142.99,GBP,1014052,https://www.scan.co.uk/products/4tb-seagate-ex...,"Seagate EXOS 4TB 3.5\"" SATA HDD/Hard Drive. De...",EXOS ST4000NM0115,None,SATA III,EXOS,4.0,NaN,NaN,6,None
4,10328354,ASUS,Asus GeForce GTX 1650 Phoenix OC 4GB Video Card,The ASUS GeForce GTX 1650 Phoenix OC 4GB Video...,219.00,AUD,1014152,https://www.auspcmarket.com.au/video-cards/nvi...,Asus GeForce GTX 1650 Phoenix OC 4GB Video Car...,GeForce GTX 1650 Phoenix OC,None,PCI-E 3.0,None,4.0,1710.0,NaN,None,None


In [20]:
# Save to json
extracted_products_1.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_1.json", orient="records", indent=2)

In [ ]:
# Run extraction for other datasets
extracted_products_2 = extractor.extract(products_2)
extracted_products_3 = extractor.extract(products_3)
extracted_products_4 = extractor.extract(products_4)

# Save to json
extracted_products_2.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_2.json", orient="records", indent=2)
extracted_products_3.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_3.json", orient="records", indent=2)
extracted_products_4.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_4.json", orient="records", indent=2)